# Example: Flux Balance Analysis of a Urea-Cycle Network

How do reaction directions and enzyme capacities constrain the rate of urea production? We will use a small metabolic network to connect the flux balance formulation from [the lecture](CHEME-5800-L6a-Lecture-FluxBalanceAnalysis-Fall-2026.ipynb) to a numerical solution.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * **Construct the reaction model:** The reaction model holds a stoichiometric matrix that describes the network topology and the steady-state balances. The model includes the urea-cycle reactions, a nitric oxide synthase branch, and exchange reactions for the inputs and outputs.
> * **Set and interpret flux bounds:** The flux bounds define the range of possible reaction rates based on thermodynamic and kinetic constraints. Combine a thermodynamic direction heuristic with enzyme-capacity estimates, keeping the units and data limitations explicit.
> * **Solve the flux balance problem:** The flux balance problem is a linear program that maximizes the urea export rate subject to the steady-state balances and flux bounds. The solution identifies the limiting enzyme capacity and the corresponding reaction fluxes.

In this example, we use a simplified urea-cycle network to calculate the maximum urea export rate and identify the enzyme capacity that limits production. Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the packages and source files used here.

Let's set up our code environment:

In [1]:
# Load packages, data paths, and lab functions -
include(joinpath(@__DIR__, "Include.jl")) # @__DIR__ locates this notebook's setup file

# Choose the plot appearance -
# After changing the theme, rerun this cell and the plotting cells.
# A single figure can override this choice with the keyword theme = :dark.
theme(:default)   # light background
# theme(:dark)   # dark background

See the [Julia documentation](https://docs.julialang.org/en/v1/) and the local [model](src/Types.jl) and [solver](src/Compute.jl) documentation for the types and calculations used here.

___

## Task 1: Build the FBA model
In this task, we will build components of the flux balance analysis (FBA) model for the urea cycle. The first step is to load the reaction network from the `Network.net` file and build an FBA model. The network file encodes the four urea cycle reactions, the nitric oxide synthase branch, and the exchange reactions in a simple text format. From this, we build a stoichiometric matrix $\mathbf{S}$, a species list, a reaction list, and a default flux bounds array.

<div>
  <center>
    <img
      src="figs/Fig-Urea-cycle-Schematic.png"
      alt="Schematic of the urea cycle"
        width="600"
    />
  </center>
</div>

> __What does the model contain?__
>
> We store all problem data in a `MyPrimalFluxBalanceAnalysisCalculationModel` instance. The key fields are:
> * `S`: the stoichiometric matrix $\mathbf{S}\in\mathbb{R}^{|\mathcal{M}|\times|\mathcal{R}|}$
> * `fluxbounds`: a $|\mathcal{R}|\times 2$ array of lower and upper bounds for each flux
> * `species` and `reactions`: ordered lists that map rows and columns of `S` to metabolite and reaction names
> * `objective`: the coefficient vector $\mathbf{c}$ for the linear objective (initially all zeros)
>
> We also return `rd::Dict{String, String}`, which maps each reaction name to its reaction string. We use it later to read the flux table.

We'll use [the `build(...)` factory method](src/Factory.jl) to construct the model from a `NamedTuple` of data. A [let block](https://docs.julialang.org/en/v1/manual/variables-and-scoping/#Let-Blocks) keeps intermediate variables private, so only `model` and `rd` are returned.

__Build the model__: Let's load the network and construct the model:

In [2]:
model, rd = let

    # Load the network -
    listofreactions = read_reaction_file(joinpath(_PATH_TO_DATA, "Network.net")); # reaction records
    S, species, reactions, rd = build_stoichiometric_matrix(listofreactions); # matrix and labels
    boundsarray = build_default_bounds_array(listofreactions); # bounds from reversibility flags

    # Build the model -
    model = build(MyPrimalFluxBalanceAnalysisCalculationModel, (
        S = S, # stoichiometric matrix
        fluxbounds = boundsarray, # lower and upper bounds; updated in Task 2
        species = species, # row labels for S
        reactions = reactions, # column labels for S
        objective = length(reactions) |> R -> zeros(R), # zero coefficients; set in Task 2
    ));

    # Return -
    model, rd
end;

___

## Task 2: Set the flux bounds and objective

In this task, we use thermodynamic estimates and enzyme capacities to set the flux bounds, then specify the urea-export objective.

> __Simplified enzyme-capacity bounds:__
>
> At the reference enzyme abundance, with saturating substrates and no allosteric regulation, the specific flux through enzyme-catalyzed reaction $j$ satisfies:
>
> $$
> -\delta_j V_{max,j}^{\circ}\leq\hat v_j\leq V_{max,j}^{\circ}.
> $$
>
> Here, $V_{max,j}^{\circ}$ is the nominal reaction capacity, with the same units as $\hat v_j$: $\mathrm{mmol\,gDW^{-1}\,h^{-1}}$. The reversibility parameter $\delta_j$ is zero for a forward-only reaction and one when both directions are allowed with equal capacities.

We first assign reaction directions, then calculate the capacities and update the model.

### Step 1: Assign reaction directions

The [recorded eQuilibrator estimates](data/urea_thermodynamics.csv) give standard transformed reaction Gibbs energies $\Delta_{\mathrm r}G_j^{\prime\circ}$ at pH 7.5, pMg 3.0, ionic strength 0.25 M, and 298.15 K. We assign directions using the heuristic:

$$
\delta_j=
\begin{cases}
1, & \Delta_{\mathrm r}G_j^{\prime\circ}>-10\,\mathrm{kJ\,mol^{-1}},\\
0, & \text{otherwise}.
\end{cases}
$$

This cutoff is a modeling assumption; physiological reversibility also depends on metabolite activities. Applying it to the recorded estimates gives:

<table style="display:table;width:100%;max-width:760px;table-layout:fixed;text-align:left;">
<colgroup><col style="width:12%;"><col style="width:44%;"><col style="width:32%;"><col style="width:12%;"></colgroup>
<thead><tr><th style="text-align:left;">Reaction</th><th style="text-align:left;">Enzyme</th><th style="text-align:right;">Δ<sub>r</sub><i>G</i><sub>j</sub><sup>′∘</sup> (kJ/mol)</th><th style="text-align:right;"><i>δ</i><sub>j</sub></th></tr></thead>
<tbody>
<tr><td>v1</td><td>Argininosuccinate synthetase</td><td style="text-align:right;">−4.3</td><td style="text-align:right;">1</td></tr>
<tr><td>v2</td><td>Argininosuccinate lyase</td><td style="text-align:right;">11.6</td><td style="text-align:right;">1</td></tr>
<tr><td>v3</td><td>Arginase</td><td style="text-align:right;">−33.9</td><td style="text-align:right;">0</td></tr>
<tr><td>v4</td><td>Ornithine transcarbamylase</td><td style="text-align:right;">−30.3</td><td style="text-align:right;">0</td></tr>
<tr><td>v5</td><td>Nitric oxide synthase</td><td style="text-align:right;">−1254.4</td><td style="text-align:right;">0</td></tr>
</tbody>
</table>

We store these assignments by reaction name in `reversibility_parameter_dictionary::Dict{String, Int}`. Exchange bounds are set separately in Step 3.

In [3]:
reversibility_parameter_dictionary = let
    
    # initialize the reversibility dictionary -
    ΔG_threshold = -10.0; # illustrative reversibility cutoff, kJ/mol
    data = CSV.read(joinpath(_PATH_TO_DATA, "urea_thermodynamics.csv"), DataFrame); # reaction names and standard transformed Gibbs energies
    
    # build the dictionary -
    # Each row contributes a reaction => δ pair; Int converts true to 1 and false to 0.
    Dict(row.reaction => Int(row.dg_prime_standard_kj_per_mol > ΔG_threshold)
         for row ∈ eachrow(data)); # fancy, what is going on here?
end;

### Step 2: Estimate enzyme capacities

We combine the [recorded turnover numbers](data/urea_turnover_numbers.csv) with a common reference enzyme abundance $e^{\circ}=0.01\,\mathrm{mmol\,gDW^{-1}}$. These inputs set an illustrative capacity scale. The records give $k_{cat,j}^{\circ}$ in $\mathrm{s^{-1}}$. To express the reaction capacity in $\mathrm{mmol\,gDW^{-1}\,h^{-1}}$, we convert seconds to hours:

$$
V_{max,j}^{\circ}
=\underbrace{\left(3600\,\mathrm{s\,h^{-1}}\right)}_{\text{time conversion}}
 k_{cat,j}^{\circ}e^{\circ}.
$$

For the nominal inputs, the capacities are:

<table style="display:table;width:100%;max-width:880px;table-layout:fixed;text-align:left;">
<colgroup><col style="width:10%;"><col style="width:18%;"><col style="width:30%;"><col style="width:42%;"></colgroup>
<thead><tr><th style="text-align:left;">Reaction</th><th style="text-align:right;"><i>k</i><sub>cat,j</sub><sup>∘</sup> (s<sup>−1</sup>)</th><th style="text-align:right;"><i>V</i><sub>max,j</sub><sup>∘</sup> (mmol gDW<sup>−1</sup> h<sup>−1</sup>)</th><th style="text-align:left;">Turnover-number source</th></tr></thead>
<tbody>
<tr><td>v1</td><td style="text-align:right;">10.00</td><td style="text-align:right;">360.00</td><td>Default based on a global median</td></tr>
<tr><td>v2</td><td style="text-align:right;">3.28</td><td style="text-align:right;">118.08</td><td>Human/duck attribution conflict</td></tr>
<tr><td>v3</td><td style="text-align:right;">190.00</td><td style="text-align:right;">6840.00</td><td>Human arginase I</td></tr>
<tr><td>v4</td><td style="text-align:right;">410.00</td><td style="text-align:right;">14760.00</td><td><i>E. coli</i></td></tr>
<tr><td>v5</td><td style="text-align:right;">1.08</td><td style="text-align:right;">38.88</td><td>Rat nitric oxide synthase</td></tr>
</tbody>
</table>

The lyase reaction `v2` has the smallest capacity among the four cycle reactions. The smaller capacity of `v5` belongs to the competing branch. The data file records the assay conditions, source publications, and the species-attribution conflict for `v2`.

The code stores the capacities in `maximum_reaction_velocity_dictionary::Dict{String, Float64}`. Changing the turnover numbers or enzyme abundance changes these values.

In [4]:
maximum_reaction_velocity_dictionary = let
    
    # initialize -
    eₒ = 0.01; # illustrative reference enzyme abundance, mmol/gDW
    seconds_per_hour = 3600.0; # convert turnover numbers from s⁻¹ to h⁻¹
    data = CSV.read(joinpath(_PATH_TO_DATA, "urea_turnover_numbers.csv"), DataFrame); # load kcat's from the data file
    
    # build the dictionary -
    # Map each reaction to its capacity in mmol/gDW/h.
    # Symbol selects the CSV column whose name contains a hyphen.
    Dict(row.reaction => row[Symbol("model_kcat_s-1")]*eₒ*seconds_per_hour
         for row ∈ eachrow(data));
end;

### Step 3: Update the flux bounds

The loop applies the enzyme-capacity bounds to `v1`–`v5`. Exchange reactions retain their bounds of $\pm1000\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$. These exchange limits do not bind at the nominal optimum, but they can constrain the solution if we increase the enzyme capacities.

The flux bounds are stored in the `fluxbounds::Array{Float64,2}` array:

In [5]:
fluxbounds = let
    
    # initialize -
    fluxbounds = copy(model.fluxbounds); # edit a separate array before attaching it to the model
    names = model.reactions; # reaction name string labels, e.g., "v1", "v2", ..., "b1", "b2", ...
    
    # main loop -
    for i ∈ eachindex(names)
        name = names[i]; # reaction label
    
        # Keep the exchange bounds -
        if startswith(name, "b") # b-prefixed labels identify exchanges in Network.net
            continue; # this means skip to the next iteration of the loop
        end
        
        VMax = maximum_reaction_velocity_dictionary[name]; # capacity, mmol/gDW/h
        δᵢ = reversibility_parameter_dictionary[name]; # 0: forward-only; 1: reversible
    
        # Apply the enzyme-capacity bounds -
        fluxbounds[i,1] = -δᵢ*VMax; # allow reverse flux only when δᵢ = 1
        fluxbounds[i,2] = VMax; # forward capacity
    end

    # Return -
    fluxbounds;
end;
model.fluxbounds = fluxbounds; # attach the updated bounds to the model

Every flux interval includes zero, so the zero-flux vector satisfies both the bounds and the steady-state balances. We now set an objective that selects a solution with urea production.

### Step 4: Set the urea-export objective

The local [solver](src/Compute.jl) maximizes $\mathbf{c}^{\top}\hat{\mathbf{v}}$, where $\mathbf{c}$ contains the objective coefficients. Since `b4` is written as `[] → M_Urea_c`, secretion corresponds to a negative flux. Setting its coefficient to $-1$ and all others to zero gives the objective:

$$
\mathbf{c}^{\top}\hat{\mathbf{v}}=-\hat v_{b4}.
$$

Maximizing this quantity maximizes urea export. We clear any previous coefficients before setting the entry for `b4`, so rerunning the cell preserves the intended objective.

In [6]:
# Set the urea-export objective -
objective = model.objective; # shared array: entry changes update the model objective
fill!(objective, 0.0); # clear previous choices when this cell is rerun
reaction_to_maximize = "b4"; # urea exchange; export has negative flux
urea_exchange_index = findfirst(==(reaction_to_maximize), model.reactions); # position of b4 in reaction order
@assert !isnothing(urea_exchange_index) "Urea exchange reaction b4 is missing"; # require a valid index before assigning the coefficient
objective[urea_exchange_index] = -1.0; # the solver maximizes -v_b4

___

## Task 3: Solve and interpret the flux balance model

In this task, we solve the linear program and examine the reaction fluxes and exchanges that support urea production.

[The solve(...) function](src/Compute.jl) returns the optimal flux vector in `solution["argmax"]` and the objective value in `solution["objective_value"]`. We report urea export as the negative of the signed `b4` flux. If the solver does not find a feasible optimum, the calculation stops with an error.


In [7]:
# Solve the linear program -
solution = solve(model); # stop here if the solver does not return a feasible optimum
urea_export_rate = -solution["argmax"][urea_exchange_index]; # convert uptake-positive flux to export rate, mmol/gDW/h

# Display the maximum export rate -
println("Maximum urea export: $(round(urea_export_rate; digits=2)) mmol/gDW/h");

Maximum urea export: 118.08 mmol/gDW/h


### Inspect the optimal fluxes

For the nominal parameters, the maximum urea export rate is $118.08\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$. The table shows each flux, its lower and upper bounds (`LB` and `UB`), and its reaction equation. All numerical columns use the same flux units; the results will change if we change the model parameters.


In [8]:
flux_table = let
    # Assemble the fluxes, bounds, and reaction equations -
    flux_bounds_array = model.fluxbounds; # rows follow model.reactions; columns are lower and upper bounds
    df = DataFrame(
        reaction = model.reactions, # keep all columns in the solver's reaction order
        flux = solution["argmax"], # optimal signed fluxes, mmol/gDW/h
        LB = flux_bounds_array[:, 1], # lower flux limits, mmol/gDW/h
        UB = flux_bounds_array[:, 2], # upper flux limits, mmol/gDW/h
        equation = [rd[r] for r in model.reactions] # look up each reaction equation by name
    );

    # Display the flux table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = [:l, :r, :r, :r, :l] # left-align text and right-align numerical columns
    );
end



┌──────────┬─────────┬─────────┬─────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ reaction │    flux │      LB │      UB │ equation                                                                                                       │
├──────────┼─────────┼─────────┼─────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ v1       │  118.08 │  -360.0 │   360.0 │ M_ATP_c+M_L-Citrulline_c+M_L-Aspartate_c = M_AMP_c+M_Diphosphate_c+M_N-(L-Arginino)succinate_c                 │
│ v2       │  118.08 │ -118.08 │  118.08 │ M_N-(L-Arginino)succinate_c = M_Fumarate_c+M_L-Arginine_c                                                      │
│ v3       │  118.08 │     0.0 │  6840.0 │ M_L-Arginine_c+M_H2O_c = M_L-Ornithine_c+M_Urea_c                                                              │
│ v4       │  118.08 │     0.0 │ 14760.0 │ M_Carbamoyl_phosphate

The four cycle reactions carry equal fluxes, while the nitric oxide synthase branch `v5` is inactive. Positive exchanges supply carbamoyl phosphate, aspartate, ATP, and water. Negative exchanges remove fumarate, urea, AMP, diphosphate, and orthophosphate. Reaction `v2` reaches its upper bound, which matches the maximum export rate.

### Check: Numerical solution
Let's check that the solver reports an optimal solution, then check the model dimensions, finite flux values, steady-state mass balances, flux bounds, and agreement between the objective value and urea export. We allow a small numerical tolerance when comparing floating-point results. Passing these checks establishes consistency with the model constraints.

In [9]:
let
    # Initialize -
    flux = solution["argmax"]; # optimal signed fluxes in model.reactions order, mmol/gDW/h
    tolerance = 1e-7; # absolute numerical tolerance, mmol/gDW/h

    # Check the solver result and model constraints -
    @testset "Urea-cycle FBA solution checks" begin
        @test solution["termination_status"] == JuMP.MOI.OPTIMAL # solver reports an optimum
        @test size(model.S) == (18, 19) # 18 species balances and 19 reaction fluxes
        @test all(isfinite, flux) # exclude NaN and infinite flux values
        
        # Check steady-state balances and both sides of each flux interval -
        @test maximum(abs.(model.S*flux)) <= tolerance # largest absolute species-balance residual
        @test all(flux .>= model.fluxbounds[:, 1] .- tolerance) # every flux satisfies its lower bound
        @test all(flux .<= model.fluxbounds[:, 2] .+ tolerance) # every flux satisfies its upper bound

        # Check the objective value and export sign -
        @test isapprox(solution["objective_value"], urea_export_rate; atol=tolerance) # objective equals -v_b4
        @test urea_export_rate >= -tolerance # export is nonnegative within numerical tolerance
    end
end;

Test Summary:                  | Pass  Total  Time
Urea-cycle FBA solution checks |    8      8  0.4s


___

## Summary

In this example, we built a urea-cycle flux balance model, set its bounds from thermodynamic and kinetic records, and solved for the maximum urea export rate.

> __Key Takeaways:__
>
> * **The network file defines the model:** We read the reaction records from the network file and built the stoichiometric matrix, the species and reaction lists, and the default bounds. The steady-state balances couple the five internal reactions, while the exchange reactions supply the inputs and remove the products.
> * **Two data sources set the bounds:** The Gibbs energy estimates and a cutoff decided which reactions could run in reverse, and the turnover numbers times a reference enzyme abundance set each capacity. These values are illustrative, so the bounds show how the pieces fit together rather than describe a measured cell.
> * **The lyase capacity sets the export rate:** Because the exchanges are written as uptake, we maximized the negative of the urea exchange flux. The maximum export rate equals the argininosuccinate lyase capacity, with all four cycle reactions carrying that flux and the nitric oxide synthase branch inactive.

We can now change the turnover numbers, enzyme abundance, or exchange bounds and see how the maximum urea export rate responds.

___